# Section 00.02B — Docker for Data Science

**Data Science and Machine Learning Course**

---

## Learning Objectives

By the end of this section you will be able to:
- Explain what Docker is and how it differs from `venv` and `conda`
- Understand images, containers, and the Dockerfile
- Write a `Dockerfile` for a data science Python project
- Build an image and run a container locally
- Run a Jupyter notebook server inside a Docker container
- Use `docker-compose` to define multi-service environments
- Pull pre-built data science images from Docker Hub
- Know when Docker is the right tool and when it is overkill

---

## 1. What is Docker?

Docker packages your code **together with the entire system it needs to run** — the OS libraries, Python version, packages, and configuration — into a single portable unit called a **container**.

Unlike `venv` or `conda`, which only isolate Python packages, Docker isolates at the operating system level. A container runs identically on your laptop, a colleague's Windows machine, a Linux server, or a cloud VM.

| Tool | Isolates | OS-level? | Portable across OSes? | Best for |
|------|----------|-----------|----------------------|----------|
| `venv` | Python packages | No | No | Local development |
| `conda` | Python + C libs | No | Partially | Data science stacks |
| **Docker** | Everything | **Yes** | **Yes** | Deployment, sharing, CI/CD |

> **Key idea:** `venv`/`conda` solve the *package* reproducibility problem. Docker solves the *full system* reproducibility problem — including the OS, system libraries, and runtime configuration.

---

## 2. Core Concepts

Understanding four terms covers 90% of day-to-day Docker use.

| Term | Analogy | Meaning |
|------|---------|--------|
| **Image** | A recipe / class | A read-only template describing the environment |
| **Container** | A running instance | A live process created from an image |
| **Dockerfile** | The recipe source code | Text file with instructions to build an image |
| **Registry** | A package index | A store for images — Docker Hub is the public default |

```
Dockerfile  →  docker build  →  Image  →  docker run  →  Container
                                  ↕
                             Docker Hub (push / pull)
```

> You build an image once and run as many containers from it as you need. Stopping a container does not delete the image.

---

## 3. Installation

**Docker Desktop** is the standard install for Windows and macOS. It bundles the Docker engine, CLI, and a GUI dashboard.

- **Windows / macOS:** download from [docker.com/products/docker-desktop](https://www.docker.com/products/docker-desktop)
- **Linux:** install the Docker Engine via the package manager — [docs.docker.com/engine/install](https://docs.docker.com/engine/install/)

> Docker Desktop requires WSL 2 on Windows. The installer handles this automatically.

In [ ]:
# Verify Docker is installed (run in your terminal)
#   docker --version
#   → Docker version 26.1.0, build ...
#
#   docker info
#   → Shows engine details, running containers, images stored
#
# Quick smoke test — download and run the hello-world image
#   docker run hello-world
#   → Downloads the image, starts a container, prints a message, exits

import subprocess
result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
print(result.stdout.strip() if result.returncode == 0 else 'Docker not found — install Docker Desktop from docker.com')

---

## 4. Writing a Dockerfile

A `Dockerfile` is a plain text file (no extension) in your project root. Each line is an instruction that adds a layer to the image.

Common instructions:

| Instruction | Purpose |
|-------------|--------|
| `FROM` | Base image to build on top of |
| `WORKDIR` | Set the working directory inside the container |
| `COPY` | Copy files from host into the image |
| `RUN` | Execute a shell command during the build |
| `ENV` | Set environment variables |
| `EXPOSE` | Document which port the container listens on |
| `CMD` | Default command to run when the container starts |

In [ ]:
# Dockerfile for a data science project — annotated

dockerfile_content = '''
# 1. Start from an official slim Python image
FROM python:3.11-slim

# 2. Set the working directory inside the container
WORKDIR /app

# 3. Copy only the requirements file first
#    (Docker caches this layer — rebuilds are faster if requirements don't change)
COPY requirements.txt .

# 4. Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# 5. Copy the rest of the project files
COPY . .

# 6. Document that the app uses port 8888 (Jupyter default)
EXPOSE 8888

# 7. Default command: start Jupyter when the container runs
CMD ["jupyter", "notebook", "--ip=0.0.0.0", "--no-browser", "--allow-root"]
'''

print(dockerfile_content)

> **Layer caching:** Docker caches each instruction. Copy `requirements.txt` and run `pip install` *before* copying your code — that way, code changes don't trigger a full reinstall.

---

## 5. Building an Image

`docker build` reads the `Dockerfile` and produces an image. The `-t` flag assigns a name and tag (`name:tag`) to the image.

In [ ]:
# Building an image (run in your terminal from the project root)
#
#  Build and tag the image
#   docker build -t my-ds-project:latest .
#   → The . tells Docker where to find the Dockerfile (current directory)
#
#  Build with a specific tag (useful for versioning)
#   docker build -t my-ds-project:v1.0 .
#
#  List all images on your machine
#   docker images
#   → REPOSITORY        TAG       IMAGE ID       SIZE
#   → my-ds-project     latest    a1b2c3d4e5f6   890MB
#
#  Remove an image
#   docker rmi my-ds-project:latest

print("docker build -t <name>:<tag> . — run from the folder containing the Dockerfile.")

---

## 6. Running a Container

`docker run` starts a container from an image. Key flags control how the container behaves.

In [ ]:
# docker run — essential flags
#
#  Run interactively with a terminal
#   docker run -it my-ds-project:latest bash
#   → Opens a bash shell inside the container
#
#  Run in the background (detached)
#   docker run -d my-ds-project:latest
#
#  Map a container port to a host port  (-p host:container)
#   docker run -p 8888:8888 my-ds-project:latest
#   → Access Jupyter at localhost:8888 in your browser
#
#  Mount a local folder into the container  (-v host_path:container_path)
#   docker run -p 8888:8888 -v $(pwd):/app my-ds-project:latest
#   → Changes to files in /app inside the container appear on your host
#
#  Pass an environment variable
#   docker run -e API_KEY=abc123 my-ds-project:latest
#
#  Remove the container automatically when it stops
#   docker run --rm my-ds-project:latest python train.py
#
#  List running containers
#   docker ps
#
#  Stop a running container
#   docker stop <container-id>

print("-v mounts your local files into the container — edits persist on your host.")

---

## 7. Running Jupyter Inside a Container

The most common data science Docker pattern: a container runs the Jupyter server, and your browser connects to it on the host. Your notebook files live on the host (mounted volume) so they persist when the container stops.

In [ ]:
# Running Jupyter in Docker — two approaches
#
# APPROACH 1: Use your own image (built from the Dockerfile in Section 4)
#   docker run --rm -p 8888:8888 -v $(pwd):/app my-ds-project:latest
#   → Open the URL printed in the terminal (includes a token)
#
# APPROACH 2: Use an official pre-built Jupyter image (no Dockerfile needed)
#   docker run --rm -p 8888:8888 -v $(pwd):/home/jovyan/work \
#     jupyter/scipy-notebook
#   → Downloads the image (~3GB) on first run, starts Jupyter immediately
#
# Official Jupyter Docker images (from jupyter/docker-stacks):
#   jupyter/base-notebook       → minimal Jupyter
#   jupyter/scipy-notebook      → + NumPy, pandas, matplotlib, scikit-learn
#   jupyter/tensorflow-notebook → + TensorFlow
#   jupyter/datascience-notebook → + R and Julia kernels
#
# Pull an image without running it
#   docker pull jupyter/scipy-notebook

print("jupyter/scipy-notebook is the fastest way to get a full DS environment in Docker.")

---

## 8. Docker Compose — Multi-service Environments

`docker-compose` (or `docker compose` in newer versions) manages multiple containers as a single application. For data science this is useful when your project needs a database, a model API, and a notebook server all running together.

Configuration lives in a `docker-compose.yml` file in the project root.

In [ ]:
# docker-compose.yml — example for a notebook + database setup

compose_content = '''
version: "3.9"

services:

  notebook:
    build: .                          # build from local Dockerfile
    ports:
      - "8888:8888"                   # expose Jupyter
    volumes:
      - .:/app                        # mount project files
    env_file:
      - .env                          # load secrets from .env
    depends_on:
      - db

  db:
    image: postgres:15                # official PostgreSQL image
    environment:
      POSTGRES_USER: analyst
      POSTGRES_PASSWORD: secret
      POSTGRES_DB: sales
    volumes:
      - db-data:/var/lib/postgresql/data   # persist DB data

volumes:
  db-data:
'''

# Common docker compose commands
#   docker compose up          → start all services
#   docker compose up -d       → start in background
#   docker compose down        → stop and remove containers
#   docker compose logs        → view logs from all services
#   docker compose build       → rebuild images

print("docker compose up starts all services defined in docker-compose.yml in one command.")

---

## 9. The `.dockerignore` File

`.dockerignore` works exactly like `.gitignore` — it prevents files from being copied into the image during `docker build`. Keeping the image lean makes builds faster and images smaller.

In [ ]:
# Typical .dockerignore for a data science project

dockerignore_content = '''
# Version control
.git/
.gitignore

# Python cache
__pycache__/
*.pyc
.ipynb_checkpoints/

# Virtual environments (not needed — Docker is the environment)
.venv/
env/

# Large data files (mount at runtime with -v instead)
data/raw/
*.csv
*.parquet

# Secrets — never bake into the image
.env

# OS files
.DS_Store
Thumbs.db

# Docker files themselves
Dockerfile
docker-compose.yml
'''

print(".dockerignore keeps the image small and prevents secrets from being baked in.")
print("Never COPY .env into an image — pass secrets at runtime with -e or --env-file.")

---

## 10. Docker Hub — Sharing Images

Docker Hub is the default public registry for Docker images — the equivalent of PyPI for packages. You can push your image so anyone can pull and run it without installing Python or any dependencies.

In [ ]:
# Pushing and pulling images — Docker Hub workflow
#
# 1. Create a free account at hub.docker.com
#
# 2. Log in from your terminal
#   docker login
#
# 3. Tag your image with your Docker Hub username
#   docker tag my-ds-project:latest yourusername/my-ds-project:latest
#
# 4. Push to Docker Hub
#   docker push yourusername/my-ds-project:latest
#
# 5. Anyone can now pull and run it
#   docker run -p 8888:8888 yourusername/my-ds-project:latest
#
# Pulling useful public data science images:
#   docker pull jupyter/scipy-notebook        # Jupyter + full DS stack
#   docker pull python:3.11-slim              # minimal Python
#   docker pull continuumio/miniconda3        # Conda base
#   docker pull pytorch/pytorch               # PyTorch with CUDA
#   docker pull tensorflow/tensorflow:latest-gpu  # TensorFlow with GPU

print("A pushed image lets anyone reproduce your exact environment with one docker pull.")

---

## 11. When to Use Docker vs venv/conda

Docker is powerful but has overhead. Choose based on the actual need.

| Situation | Best tool |
|-----------|----------|
| Local exploratory analysis | `conda` or `venv` |
| Sharing a notebook with a colleague | `requirements.txt` or `environment.yml` |
| Deploying a model as an API | **Docker** |
| Reproducibility across different OSes | **Docker** |
| CI/CD pipelines (GitHub Actions, etc.) | **Docker** |
| Needing a specific system library (CUDA, GDAL) | **Docker** |
| Quick experiment on your own machine | `conda` or `venv` |
| Teaching / bootcamp exercises | `conda` or `venv` |

> **Rule of thumb:** Use `conda`/`venv` while developing. Switch to Docker when you need to *ship* the work to another machine, a server, or a cloud service.

In [ ]:
# Quick reference — most used Docker commands
#
#  IMAGES
#  docker images                          → list local images
#  docker build -t name:tag .             → build from Dockerfile
#  docker pull image:tag                  → download from registry
#  docker rmi name:tag                    → delete an image
#
#  CONTAINERS
#  docker run -it name bash               → run interactively
#  docker run -d -p 8888:8888 name        → run in background, expose port
#  docker run --rm -v $(pwd):/app name    → mount folder, auto-remove on exit
#  docker ps                              → list running containers
#  docker ps -a                           → list all containers (inc. stopped)
#  docker stop <id>                       → stop a container
#  docker rm <id>                         → delete a stopped container
#  docker logs <id>                       → view container output
#  docker exec -it <id> bash             → open a shell in a running container
#
#  COMPOSE
#  docker compose up -d                   → start all services in background
#  docker compose down                    → stop and remove all services
#  docker compose logs -f                 → follow logs from all services

print("docker exec -it <id> bash — the most useful debugging command.")

---

## 12. Mini Exercise — Containerise a Notebook Project

Build and run a Jupyter container for a minimal data science project.

In [ ]:
# Full walkthrough (run in your terminal)
#
# 1. Create a project folder
#   mkdir docker-practice && cd docker-practice
#
# 2. Create requirements.txt
#   echo "pandas" > requirements.txt
#   echo "matplotlib" >> requirements.txt
#   echo "jupyter" >> requirements.txt
#
# 3. Create a Dockerfile
#   (paste the content from Section 4)
#
# 4. Create a .dockerignore
#   echo ".git" > .dockerignore
#   echo "__pycache__" >> .dockerignore
#
# 5. Build the image
#   docker build -t ds-practice:latest .
#
# 6. Run Jupyter from the container, mounting current folder
#   docker run --rm -p 8888:8888 -v $(pwd):/app ds-practice:latest
#
# 7. Open the URL printed in the terminal — Jupyter opens in your browser
#    Create a notebook, import pandas, save it — the file appears on your host
#
# SHORTCUT — skip steps 2-5, use an official image directly:
#   docker run --rm -p 8888:8888 -v $(pwd):/home/jovyan/work \
#     jupyter/scipy-notebook

print("After the exercise: Jupyter running in Docker, files persisted on your host.")

---

## Key Takeaways

- Docker isolates at the **OS level** — the container carries everything: Python, system libraries, and configuration. It runs identically anywhere.
- The three core objects: **Dockerfile** (recipe) → **Image** (built snapshot) → **Container** (running instance).
- Copy `requirements.txt` and run `pip install` *before* copying your code in the Dockerfile — Docker's layer cache makes rebuilds fast.
- `-p host:container` maps a port; `-v host_path:container_path` mounts a folder so files persist after the container stops.
- Never bake secrets into an image — pass them at runtime with `-e KEY=value` or `--env-file .env`.
- `docker compose up` starts an entire multi-service environment (notebook + database, etc.) with one command.
- Use `conda`/`venv` during development; switch to Docker when you need to ship the environment to another machine, a server, or a cloud pipeline.
- `jupyter/scipy-notebook` from Docker Hub gives a full data science Jupyter environment with zero configuration.

---

*Next: **Section 00.03 — Jupyter Overview** — now that your environment is reproducible and portable, explore the notebook interface in depth.*